In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Add `python modules/` to Python path
sys.path.append(os.path.join('..', 'python modules'))

from feature_processor import FeatureProcessor

# Pandas display options
pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 150)

# Paths
data_dir = '../data'  # from the notebooks directory
raw_hdf5_path = os.path.join(data_dir, 'global_data_Trucost_Compustat_merged_universal_features.h5')
processed_hdf5_path = os.path.join(data_dir, 'processed_feature_data.h5')

print('Environment setup complete. FeatureProcessor imported.')


# Feature Engineering Pipeline

## Objective

Construct a high-quality feature set for corporate carbon-emission prediction.

## Main steps

1. Environment setup and data loading
2. Data reconstruction and validation
3. (Optional) variable inventory for manual screening
4. (Optional) drop manually-flagged features
5. Run the (GPU-first) feature-engineering pipeline
6. Validate outputs and save processed artifacts
7. (Optional) feature-importance exploration

---

# 1. Environment setup and data loading

This notebook assumes the universal HDF5 file has already been created in `../data/`.



In [ ]:
# Code block 2 (updated): Load data from the universal (NumPy-based) HDF5 and reconstruct a DataFrame

import h5py
import time

print('Loading data from the universal HDF5 file and reconstructing a DataFrame...')
print(f'Target file: {raw_hdf5_path}')

# IMPORTANT: label names must match the names used when creating the HDF5 file
ORIGINAL_LABEL_NAMES = [
    'Scope1', 'Scope2', 'Scope3_upstream', 'Scope3_prod', 'Scope3_downLA', 'Scope_total'
]
LOG_LABEL_NAMES = [f'log_{col}' for col in ORIGINAL_LABEL_NAMES]

start_time = time.time()
try:
    with h5py.File(raw_hdf5_path, 'r') as hf:
        # 1) Numeric features
        print(' - Reading numeric feature matrix and feature names...')
        features_matrix = hf['features'][:]
        feature_names = [name.decode('utf-8') for name in hf['feature_names'][:]]
        df_numeric = pd.DataFrame(features_matrix, columns=feature_names)

        # 2) Non-numeric data
        print(' - Reading non-numeric data and column names...')
        non_numeric_matrix = hf['non_numeric_data'][:]
        non_numeric_names = [name.decode('utf-8') for name in hf['non_numeric_feature_names'][:]]
        df_non_numeric = pd.DataFrame(non_numeric_matrix, columns=non_numeric_names)

        # 3) Labels (original + log)
        print(' - Reading original and log-transformed labels...')
        labels_original_np = hf['labels_original'][:]
        labels_log_np = hf['labels_log'][:]
        df_labels_orig = pd.DataFrame(labels_original_np, columns=ORIGINAL_LABEL_NAMES)
        df_labels_log = pd.DataFrame(labels_log_np, columns=LOG_LABEL_NAMES)

    # 4) Concatenate into a single DataFrame
    print(' - Concatenating into a single DataFrame...')
    df_raw = pd.concat([df_non_numeric, df_numeric, df_labels_orig, df_labels_log], axis=1)

    end_time = time.time()
    print('Data loaded and reconstructed successfully.')
    print(f'Elapsed time: {end_time - start_time:.2f} seconds')
    print(f'Reconstructed df shape: {df_raw.shape}')

    # Ensure bytes/object columns are converted to strings for display
    for col in df_raw.select_dtypes(include=['object']).columns:
        df_raw[col] = df_raw[col].astype(str)

    display(df_raw.head())

    # Sanity check: ensure key columns exist
    required_cols_check = ['gvkey', 'fiscalyear', 'loc', 'GICSSector'] + LOG_LABEL_NAMES
    missing_cols = [col for col in required_cols_check if col not in df_raw.columns]
    if not missing_cols:
        print('[OK] Key columns are present: gvkey, fiscalyear, loc, GICSSector, and log labels.')      
    else:
        print(f"[WARNING] Missing key columns: {missing_cols}")

except FileNotFoundError:
    print(f"[ERROR] HDF5 file not found: {raw_hdf5_path}")
    print('Please verify the upstream data-ingestion step and file path.')
except Exception as e:
    print(f"Unexpected error during loading/reconstruction: {e}")


# 2. Data Reconstruction and Validation

Load data from HDF5 and reconstruct a full DataFrame

In [ ]:
# ============================================================================
# Variable inventory (optional)
# ============================================================================
# This section was used to produce detailed variable reports for manual screening.
# For the submission version, we keep a lightweight summary.

print('Variable inventory (lightweight summary)')
print(f'Total columns in df_raw: {df_raw.shape[1]:,}')

# Candidate numeric feature columns (excluding labels)
label_cols = set(ORIGINAL_LABEL_NAMES + LOG_LABEL_NAMES)
num_cols = [c for c in df_raw.columns if pd.api.types.is_numeric_dtype(df_raw[c]) and c not in label_cols]
print(f'Numeric candidate features: {len(num_cols):,}')
print('First 50 numeric feature names:')
print(num_cols[:50])


# 3. Variable analysis and interpretation

## 3.1 Comprehensive variable inventory (optional)

The lightweight inventory above is sufficient for reproducing the feature-engineering pipeline.
If you need the original exhaustive inventory/export workflow, refer to the project history or saved reports in `related files/`.



In [ ]:
# ============================================================================
# Optional: export variable report to Excel
# ============================================================================
# This section is intentionally omitted in the submission version.
# If needed, export df_raw.columns / dtypes / missing rates to an Excel file for manual review.

print('Optional Excel export is omitted in this submission-oriented notebook.')


## 3.2 Excel report generation

Generate a detailed variable-analysis Excel report (commented out; enable if needed)

In [ ]:
# ============================================================================
# Drop manually-flagged features (optional)
# ============================================================================

import pathlib

# Default: start from df_raw
_df_in = df_raw

# If a manual screening file exists, use it to drop columns.
# Example screening files live in ../related files/.
related_dir = pathlib.Path('..') / 'related files'
flag_files = [
    related_dir / 'marked_variable_analysis_report_404vars_20251105_200105.csv',
]

df_filtered = _df_in

dropped = []
for f in flag_files:
    if f.exists():
        try:
            flags = pd.read_csv(f)
            # Heuristic: look for a boolean/marker column indicating a feature should be dropped.
            # If your screening file uses a different schema, adjust here.
            candidate_cols = [c for c in flags.columns if c.lower() in ('drop', 'remove', 'marked_drop', 'marked_remove')]
            name_cols = [c for c in flags.columns if c.lower() in ('variable', 'feature', 'name', 'column')]
            if candidate_cols and name_cols:
                drop_col = candidate_cols[0]
                name_col = name_cols[0]
                to_drop = flags.loc[flags[drop_col].astype(str).str.lower().isin(['1','true','yes','y']), name_col].astype(str).tolist()
                to_drop = [c for c in to_drop if c in df_filtered.columns]
                if to_drop:
                    df_filtered = df_filtered.drop(columns=to_drop)
                    dropped.extend(to_drop)
                    print(f'Dropped {len(to_drop)} columns based on {f.name}.')
                else:
                    print(f'Found {f.name} but no matching columns to drop in df_raw.')
            else:
                print(f'Found {f.name} but could not infer drop schema; skipping.')
        except Exception as e:
            print(f'Failed to read/apply flags from {f.name}: {e}')

print(f'Input df shape: {df_raw.shape} -> filtered df shape: {df_filtered.shape}')
print(f'Total dropped columns: {len(set(dropped))}')


# 4. Feature screening and removal

## 4.1 Load screening flags and drop invalid features (optional)

If you have a manually curated screening file, the previous code cell can apply it to drop columns before feature engineering.



In [ ]:
# ============================================================================
# Run the GPU-first feature-engineering pipeline
# ============================================================================

from gpu_feature_processor import FeatureProcessor as GPUFeatureProcessor

fp_gpu = GPUFeatureProcessor(
    top_n_features=100,
    correlation_threshold=0.95,
    missing_threshold=0.3,
    use_gpu=True,
)

df_processed_gpu = fp_gpu.process(df_filtered)
print('Feature engineering completed.')
print('Processed df shape:', df_processed_gpu.shape)


# 5. GPU-based feature engineering

## 5.1 Execute the feature-engineering pipeline

The pipeline performs screening, imputation, outlier handling, and LightGBM-based Top-N feature selection.



In [ ]:
# ============================================================================
# Final validation and summary
# ============================================================================

print('Final validation summary')
print('-' * 60)
print(f'Processed df shape: {df_processed_gpu.shape}')

id_cols = [c for c in ['gvkey', 'fiscalyear', 'loc', 'GICSSector'] if c in df_processed_gpu.columns]
label_cols = [c for c in df_processed_gpu.columns if c.startswith('log_') or c in ORIGINAL_LABEL_NAMES]
feature_cols = [c for c in df_processed_gpu.columns if c not in set(id_cols + label_cols)]

print(f'ID columns: {len(id_cols)} -> {id_cols}')
print(f'Label columns: {len(label_cols)}')
print(f'Feature columns: {len(feature_cols)}')

# Expose for downstream notebooks
FEATURE_COLS = feature_cols
TARGET_COLS = [c for c in df_processed_gpu.columns if c.startswith('log_')]


# 6. Result validation and saving

## 6.1 Final validation and summary

We summarize the processed dataset and export it to `../data/processed_feature_data.h5` for downstream experiments.



In [ ]:
# Validate missingness distribution (features vs labels)
print('=== Missingness validation ===')

missing_rate = df_processed_gpu.isnull().mean().sort_values(ascending=False)
print('Top-20 columns by missing rate:')
print(missing_rate.head(20))

print('Missing rate summary (mean over columns):')
print(missing_rate.mean())


## 6.2 Missingness validation

Analyze missingness distribution after processing

In [ ]:
# Save processed data to a new HDF5 file
print(f'Saving processed data to: {processed_hdf5_path} ...')

# NOTE: This writes an artifact file only when executed.
# It does NOT modify any existing raw data files.

df_processed_gpu.to_hdf(
    processed_hdf5_path,
    key='processed_features',
    mode='w',
    complevel=9,
    complib='blosc',
)

print('Saved processed HDF5 successfully.')


## 6.3 Data saving

The processed dataset is stored in an HDF5 file for efficient loading by subsequent experiment notebooks.



In [ ]:
# ============================================================================
# Optional: inspect selected feature importance (if available)
# ============================================================================

# The GPUFeatureProcessor stores feature names and importance scores when LightGBM selection succeeds.
# We show a quick Top-20 preview.

names = getattr(fp_gpu, 'feature_names_', None)
imps = getattr(fp_gpu, 'feature_importance_', None)

if names is None or imps is None:
    print('No feature-importance artifacts found on fp_gpu (selection may have used a fallback).')
else:
    imp_s = pd.Series(imps, index=names).sort_values(ascending=False)
    print('Top-20 selected features by importance:')
    print(imp_s.head(20))


# 7. Feature-importance analysis (optional)

## 7.1 LightGBM feature-importance view

This section provides a lightweight view of the selected features and their importance scores.



In [ ]:
# ============================================================================
# Feature-importance summary (optional)
# ============================================================================

if 'FEATURE_COLS' in globals():
    print(f'# selected feature columns: {len(FEATURE_COLS)}')
    print('First 20 features:')
    print(FEATURE_COLS[:20])


## 7.2 Load historical LightGBM feature-importance results (optional)

If you have saved importance results from earlier runs, you can load them here for deeper exploration.



In [ ]:
# ============================================================================
# Load historical LightGBM feature-importance results (optional)
# ============================================================================

import pathlib

results_dir = pathlib.Path('..') / 'results'

# Example: look for a saved importance file if you maintain one.
# Adjust the pattern to your repository naming convention.
candidates = sorted(results_dir.glob('**/*importance*.pkl'))

if not candidates:
    print('No historical importance files found under ../results/.')
else:
    print('Found historical importance files (showing up to 5):')
    for p in candidates[:5]:
        print(' -', p)
